# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [2]:
import pandas as pd
import numpy as np
import os
from glob import glob

import tensorflow as tf
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten, TimeDistributed, Reshape
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm

In [3]:
input_path = [i for i in glob('Dataset\*\*\*.mp4')]
# First removing the path by split and then splitting the string to get the label
output_label = [(i.split("\\")[-1]).split(" ")[-1] for i in glob('Dataset\*\*')] 

print(output_label)
input_path

['loud', 'quiet', 'happy', 'sad', 'Beautiful', 'Ugly', 'Deaf', 'Blind']


['Dataset\\Adjectives\\1. loud\\MVI_5177.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5178.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5179.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5257.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5258.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5259.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5335.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5336.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_5337.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9289.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9290.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9291.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9368.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9369.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9370.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9448.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9449.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9450.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9534.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9535.mp4',
 'Dataset\\Adjectives\\1. loud\\MVI_9536.mp4',
 'Dataset\\Ad

In [15]:
labels = pd.Series(output_label).unique()
labels = pd.Series(labels).to_list()

train_path = pd.Series(input_path)

train_path.sample(5),labels

(83         Dataset\Adjectives\6. Ugly\MVI_9844.mp4
 0          Dataset\Adjectives\1. loud\MVI_5177.mp4
 71    Dataset\Adjectives\5. Beautiful\MVI_9569.mp4
 93         Dataset\Adjectives\7. Deaf\MVI_9850.mp4
 70          Dataset\Adjectives\4. sad\MVI_9722.mp4
 dtype: object,
 ['loud', 'quiet', 'happy', 'sad', 'Beautiful', 'Ugly', 'Deaf', 'Blind'])

In [ ]:
label_map = dict()

for i in range(len(labels)):
    for j in input_path:
        sep = j.split("\\")
        cur_label  = sep[-2].split(" ")[-1]
        if labels[i] in cur_label:
            label_map[cur_label] = label_map.get(cur_label, 0) + 1
            
label_map    

{'loud': 21,
 'quiet': 21,
 'happy': 21,
 'sad': 8,
 'Beautiful': 8,
 'Ugly': 8,
 'Deaf': 8,
 'Blind': 8}

In [30]:
# Loading all the video frames
video_frames = []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    window = []
    
    for video in label_videos:
        # table = pq.read_table("MP_data/" + f"{label}/" + video)

        # df = table.to_pandas()

        # numpy_array = df.to_numpy()
        
        res = np.load("MP_data/" + f"{label}/" + video,allow_pickle=True)
        
        # window.append(numpy_array)
        window.append(res)
        
    video_frames.append(window)

print(len(video_frames))

        

  0%|          | 0/8 [00:00<?, ?it/s]

8


In [31]:
# Padding vedio sequences so all array are of equal length


# Checking whether the gpu is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        print("Using GPU:", gpus)

    except RuntimeError as e:
        print(e)

else:
    print("No GPU found.")




No GPU found.


In [32]:


# Pad sequences to ensure they have the same length
max_length = max(len(seq) for seq in video_frames)
padded_video_frames = []

for frames in video_frames:
    padded_video_frames.append(pad_sequences(frames, maxlen=max_length, dtype='float32', padding='post', truncating='post'))

for i, frames in enumerate(padded_video_frames):
    print(f"Video {i+1} shape: {frames.shape}")

Video 1 shape: (21, 21, 1662)
Video 2 shape: (21, 21, 1662)
Video 3 shape: (21, 21, 1662)
Video 4 shape: (8, 21, 1662)
Video 5 shape: (8, 21, 1662)
Video 6 shape: (8, 21, 1662)
Video 7 shape: (8, 21, 1662)
Video 8 shape: (8, 21, 1662)


In [34]:
max_frames = max(video.shape[0] for video in padded_video_frames)

# Pad the number of frames for each video
X = np.zeros((len(padded_video_frames), 21, max_frames, 1662), dtype='float32')

for i, video in enumerate(padded_video_frames):
    X[i, :video.shape[0]] = video

print(X.shape)

(8, 21, 21, 1662)


In [35]:
y = np.array([label_map[label] for label in labels])
y = to_categorical(y).astype(int)

print(y.shape)

(8, 22)


In [36]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) 

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)



X_train shape: (6, 21, 21, 1662)
X_val shape: (2, 21, 21, 1662)
y_train shape: (6, 22)
y_val shape: (2, 22)


# Model

## Architecture

In [42]:
tf.keras.backend.clear_session()

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

class INCLUDE_V1(Model):
    def __init__(self, input_shape, num_classes=50, time_steps=19):
        super().__init__()
        self.num_classes = num_classes
        self.input_shape_ = input_shape

        self.input_layer = Input(shape=input_shape)
        
        
        # Bidirectional LSTM layers
        self.BD1 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD2 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD3 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD4 = Bidirectional(LSTM(64, return_sequences=True))
        
        # Flatten the output
        self.flatten = Flatten()
        
        # Fully connected layer
        self.dense = Dense(128, activation='relu')
        
        # Output layer
        self.outputs = Dense(self.num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.BD1(inputs)
        x = self.BD2(x)
        x = self.BD3(x)
        x = self.BD4(x)
        
        x = self.flatten(x)
        x = self.dense(x)
        
        return self.outputs(x)



In [45]:
# Initializing the model and inputing the input shape 

# input_shape = X_train[0].shape #(21, 19, 1662)
input_shape = (21, 19, 1662)
num_classes = 8
batch_size = 8

inputs = Input(shape=input_shape)

model = INCLUDE_V1(input_shape=input_shape, num_classes=num_classes)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
# model.build(input_shape=(None, *input_shape))
model.build(input_shape=(input_shape))

model.summary()


c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\layer.py:361: UserWarning: `build()` was called on layer 'include_v1_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "include_v1_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_8 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.fit(X_train,
          y_train,
          validation_data=(X_val, y_val), 
          epochs=10, 
          batch_size=batch_size)


Epoch 1/10


ValueError: Exception encountered when calling INCLUDE_V1.call().

[1mInput 0 of layer "bidirectional_8" is incompatible with the layer: expected ndim=3, found ndim=4. Full shape received: (None, 21, 21, 1662)[0m

Arguments received by INCLUDE_V1.call():
  • inputs=tf.Tensor(shape=(None, 21, 21, 1662), dtype=float32)
  • training=True

In [ ]:
# input_shape = ()
# input_shape = (1920, 1080, 3)
# time_steps = 19

# target_height, target_width = 224, 224

# input_layer = tf.zeros((batch_size, time_steps, *input_shape))
# print("Original input shape:", input_layer.shape)

# downsampled_input = downsample_frames(input_layer, target_height, target_width)
# print("Downsampled input shape:", downsampled_input.shape)

# model.build(input_shape=downsampled_input)
